# openpilot supercombo on KITScenes: one scene, every keyframe

> **Outputs are not committed: run the notebook to see them.** Every figure, table and number the text refers to comes from a run, and git keeps none of them: `tests/test_notebooks_are_bare.py` holds this file to that.
>
> 1. **Once:** `make setup`, then `uv pip install --python .venv/bin/python -e ".[analysis]"` for Jupyter and what the notebook draws with.
> 2. **The site:** `paddock.toml` (or `PADDOCK_CONFIG`) says where `{runs}` and the data are. It reads `RUN_ID`'s records (§1; part 3 checks against them) under `{runs}`, and the KITScenes data for `SCENE`.
> 3. **Run it:** run all on the repo's `.venv` kernel, or headless from the repo root: `.venv/bin/jupyter execute --output='{notebook_name}_all_executed' tools/models/openpilot/analysis/openpilot_drv_supcombo_kitscenes_scene_sweep.ipynb`. That writes an executed copy beside this file, which git ignores. Never `--inplace`: it writes the outputs into this file, and `make check` fails on that. A run in Jupyter or VS Code saves them here too; strip it before committing, with `.venv/bin/python -m tools.strip_notebooks tools/models/openpilot/analysis/openpilot_drv_supcombo_kitscenes_scene_sweep.ipynb`.
>
> **What it needs.** openpilot's checkout at the sha `repos.yaml` pins, with its own environment set up (`paddock doctor` says whether it is), because the notebook starts image nodes ([`openpilot_drv_supcombo_image_node.py`](openpilot_drv_supcombo_image_node.py), which must stay beside it) in openpilot's interpreter. By default 16 of them on a GPU (`DEVICE`, `NODES`, §1): the GPUs are shared, so check `nvidia-smi` first, or set `DEVICE = "cpu"`; a replay costs about 1 s on either. Each node holds about 7 GB of frames in host memory (§1), about 112 GB at the default 16. Results go under `{runs}/kitscenes/_analysis/openpilot/scene_sweep/`.

The benchmark scores each scene only at the keyframes its shared sample list names: 40, 130, 220, and so on, 9 s apart. The [image-tweak notebook](openpilot_drv_supcombo_kitscenes_image_tweak.ipynb) knows only those keyframes, which is why `FOCUS = "dbba76d9/150"` raises a `KeyError` there. Frame 150 is in the scene, but no listed sample starts at it.

This notebook makes every frame of one scene a keyframe and runs openpilot on each, built the way the list builds its own rows (the route is read from where the car is 5 s later; more than 5 m to a side counts as a turn). For each keyframe it gives:

1. **The route**, read by the list's rule, beside the listed rows. On the listed keyframes the two must agree.
2. **Every score and every head of the model.** ADE, FDE, lateral and longitudinal error, heading, speed, the camera height the model reads, desire, lead, lane lines and plan sigma. Each comes with a noise floor: the largest change that any of eight half-pixel shifts of the image makes.
3. **A check that "stock here" is the recorded model**, on the listed keyframes.
4. **A timeline** of all of the above across the scene.
5. **Every keyframe's figure**, written as JPEG to a cache directory outside the repo.
6. **A frame-by-frame view**: one keyframe's figure and the score rows around it.

Each keyframe is its own 4.8 s replay from a reset model, the way the benchmark scores it; it is not one continuous drive. Keyframes k and k + 1 share 47 of their 48 history frames. A jump between them is the model's response to one more frame, not state carried over.

The code is in [the sweep lib](openpilot_drv_supcombo_kitscenes_scene_sweep_lib.py). It is the image-tweak lab with more keyframes: it is imported, not copied, so what a `CONFIG` does to the image is defined in one place. It runs stock, and beside it §1's `CONFIG`, which as committed is the configuration the [config-run notebook](openpilot_drv_supcombo_kitscenes_config_run.ipynb) reads; `CONFIG = None` runs stock alone.

In [ ]:
import sys

import pandas as pd

from paddock.config import REPO_ROOT

sys.path.insert(0, str(REPO_ROOT / "tools/models/openpilot/analysis"))  # the libs, beside this notebook
import openpilot_drv_supcombo_image_tweak_lib as T
import openpilot_drv_supcombo_kitscenes_scene_sweep_lib as S

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

## 1. Parameters

- **`SCENE`**: a scene id, or a unique prefix of one.
- **`START`, `STOP`, `STRIDE`**: the keyframes, as source frame indices at 10 Hz.
    - `STOP = None` means the last keyframe with the benchmark's full 4 s of GT (8 poses at 0.5 s).
    - Keyframes from 40 frames before that point onward read their route less than 5 s ahead, because the scene ends first. The `lookahead_s` column says how far ahead it was read.
- **`RUN_ID`**: the recorded stock run that part 3 checks against.
- **`DEVICE`**: `"cuda"` or `"cpu"`.
    - Either costs about 1 s per replay. Measured on the same 16 keyframes: a median of 1.0 s on a GPU and 1.1 s on the CPU. The time goes to the 98 ticks of frame preparation, which run on the CPU either way.
    - `"cuda"` is the default because the recorded run was made on a GPU, so part 3's check can be exact.
    - The GPUs are shared, so check `nvidia-smi` first. Each node takes well under 1 GB.
- **`NODES`**: the number of model processes. Keyframes are split into contiguous blocks, one per node, so each node decodes its frames once.
- **`HEIGHT`, `OVERRIDES`, `RECOMPUTE`**: as in the image-tweak notebook.
- **`CONFIG`**: `None` runs stock. Otherwise give a dict of the image-tweak knobs; it is run beside stock, or beside `AGAINST`, on every keyframe.
- **`OVERLAY`**: where the GT and plans are drawn in each keyframe's figure: `"img"`, `"big_img"`, `"camera"`.
- **`WIDGETS`**: a slider in part 6. It is off by default because VS Code cannot draw a saved widget.
- **`WORKERS`**: the processes that draw the figures in part 5.

In [ ]:
SCENE = "dbba76d9-91ec-b44f-f3d3-6348d55f7849"  # the "not planned" left turn of the image-tweak notebook
START, STOP, STRIDE = 40, None, 1
RUN_ID = "20260930T154018-242869"  # the open-loop notebook's stock re-run
DEVICE = "cuda"  # as fast as "cpu" here, and the recorded run's device
NODES, THREADS = 16, 8  # the speed is in the nodes: ~1 s a replay each, ~7 GB of frames each
HEIGHT = "context"
OVERRIDES = {}
RECOMPUTE = False
# CONFIG runs beside stock, and None runs stock alone. As committed it is the configuration the config-run
# notebook reads, so a default run sweeps that configuration and stock.
CONFIG = dict(
    # geometry
    height_scale=1.22,  # s: the road as from a camera at h / s
    zoom=1.0,  # k: magnified k x
    hstretch=1.0,  # g: stretched g x sideways
    yaw_deg=0.0,  # the view turned left (deg)
    turn_yaw_deg=0.0,  # the view turned toward the route's side (deg); negative is away
    pitch_deg=-1.7,  # the view tilted up (deg)
    roll_deg=0.0,  # the view rolled, its right side down (deg)
    ahead_m=0.0,  # the camera moved forward (m)
    left_m=0.0,  # and left (m)
    dx_px=0.0,  # the sampling grid moved right (model pixels)
    dy_px=0.0,  # 30.0,  # and down
    views="both",  # where the geometry applies: "both", "img" or "big_img"
    # sampling
    supersample=1,  # n: n x n camera samples per model pixel
    ss_views="big_img",  # where supersampling applies
    sharpen=0.0,  # alpha; -1 is a 3 x 3 blur
    # tone, on both views
    gamma=1.0,  # on luma; under 1 is brighter
    contrast=1.0,  # about mid-grey
    brightness=0.0,  # luma steps
    saturation=1.0,  # 0 is grey
    # ablation
    blank="",  # "img" or "big_img": that view shown as flat grey
    # the source
    camera="base",  # "ring" (stock) or "base"
    fov_deg=55.0,  # cut the frame to this many degrees across; T.EON_FOV is comma's 65.2; 0 is as recorded
    fov_aspect=T.EON_W / T.EON_H,  # the cut's width over height
    fov_width=T.EON_W,  # resample the cut to this many px across; 0 crops whole pixels
    fov_filter="area",  # "area", "bilinear", "lanczos" or "nearest"
    fov_told_deg=0.0,  # tell the model the cut is this wide (deg); 0 tells it the truth
    # the intent
    desire="turn",  # "off", "turn" or "lane_change"
    desire_lead_s=0.0,  # the desire starts this long before the keyframe (s), 0 to 4.8
)
AGAINST = None
OVERLAY = ("img", "big_img", "camera")
WIDGETS = False
WORKERS = 24

S.WIDGETS = WIDGETS
lab = S.SceneLab(
    SCENE,
    RUN_ID,
    start=START,
    stop=STOP,
    stride=STRIDE,
    device=DEVICE,
    nodes=NODES,
    threads=THREADS,
    height=HEIGHT,
    overrides=OVERRIDES,
    recompute=RECOMPUTE,
)

## 2. The keyframes and their routes

`route` is the route the keyframe is scored with. On a listed keyframe it is the list's row; otherwise the list's rule makes it. `by the rule` is the rule's reading on every keyframe, and `same` must be True on the listed ones, or this notebook's keyframes are not built the benchmark's way.

`gt_dpsi_l` is how far the GT turns in 3 s (degrees, left +), and `gt_y_3s` is how far left it is at 3 s (m). Together they show where the turn actually happens, as opposed to where the 5 s rule first calls it.

In [ ]:
routes = lab.route_table()
assert routes.loc[lab.listed, "same"].all(), "the rule does not reproduce the listed rows"
routes.round(2)

## 3. Stock on every keyframe

Stock on every keyframe, plus the eight half-pixel shifts on each: 9 × the number of keyframes runs, cached by sample, views, device and code. `noise_<col>` is the largest change that any shift makes to that column. A difference smaller than it, between keyframes or between configs, is nothing.

The columns are grouped like this:

- **The keyframe:** `t_s`, `listed`, `route`, `side`, `lookahead_s`, `v` (the pose speed).
- **The plan against the GT:** `ade`, `fde_1s`/`2s`/`3s`, `lat` and `lon` (the 3 s error split across and along the GT), `path_err` (off-path, regardless of timing), `ade_long` (to 10 s, wherever there is GT).
- **The plan's shape:** `dpsi_l` (its heading change over 3 s, left +), `gt_dpsi_l` (the GT's), `dpsi_l_err`, and `dpsi_10s_l`. `dpsi`, `gt_dpsi` and `turn_frac` are the same measures taken toward the route's side; they are NaN on a straight route, where `mode` says "straight route".
- **Speed:** `v0_plan` (the plan's first step), `v0_gt`, `v0_ratio`, `plan_v0`, `head` (the pose head's speed), `head_ratio`, and `speed_rmse`.
- **The model's other heads:**
    - `cam_h` (the camera height the road head reads), and `road_pitch_deg` and `road_yaw_deg`;
    - the desire probabilities: `p_none`, `p_left`, `p_right`, `p_change_l`, `p_change_r`, and `p_turn` (toward the route);
    - the lead: `lead_prob`, `lead_d`, `lead_v_rel`;
    - `line_p_*` (the four lane lines' probabilities, left to right);
    - `sigma_*_3s` (the plan's uncertainty at 3 s).
- **`in_img`, `in_big_img`:** the share of each view's pixels that come from inside the camera image.

In [ ]:
scores = lab.sweep(CONFIG, AGAINST)
scores

In [ ]:
scores["ade"].mean()

**Is "stock here" the recorded model?** On the listed keyframes, the largest gap between any waypoint planned here and the recorded run's within 3 s must stay under `T.TOL_M` (0.25 m). The recorded run was made on a GPU, so on `"cuda"` the gap should be zero.

In [ ]:
lab.reproduce()

## 4. The timeline

Every score and head across the scene, one panel per group. The grey band is the baseline's noise floor. The dotted verticals are the listed keyframes, and the shading marks the keyframes whose route is a turn (blue for left, red for right).

In [ ]:
plotted = lab.timeline(CONFIG, AGAINST)  # what the panels drew, one row per keyframe and arm
plotted

In [ ]:
plotted["ade"].mean()

**Has the config improved the numbers?** `plotted` holds every value the timeline drew, one row per keyframe and arm. Its columns:

- the keyframe: `t_s`, `listed`, `route`, `side`;
- each drawn column, and the references drawn beside them: `gt_dpsi_l`, `v`, `v0_gt`;
- three errors against those references (absolute, so lower is closer):
    - `heading_err`: the plan's 3 s heading against the GT's;
    - `v0_err`: the plan's first-step speed against the GT's;
    - `cam_h_err`: the read camera height against the map's;
- `noise_<col>` on the baseline's rows.

`lab.compare(plotted)` pairs config with baseline keyframe by keyframe. For each column it gives:

- each arm's mean and median;
- the difference's mean and median, with a 95% interval for the mean;
- how many keyframes moved beyond the half-pixel noise floor, in each direction;
- a verdict: "better" or "worse" for error columns when the interval excludes 0, or "up" or "down" for the rest.

The interval comes from a moving-block bootstrap over 1 s blocks. Neighbouring keyframes share 47 of their 48 history frames, so 120 keyframes are far fewer independent samples than they look.

`ks` restricts the comparison to part of the scene, for example `slice(110, 137)` for the stretch where stock plans straight through the turn. With `CONFIG = None` the baseline is compared with itself, and every difference is 0.

In [ ]:
diffs = lab.compare(plotted)  # the whole scene
diffs_turn = lab.compare(plotted, ks=slice(110, 137))  # the turn stock misses
display(diffs, diffs_turn)

## 5. Every keyframe's figure, to the cache

Each keyframe's figure is drawn as `kf_<k>.jpg` into the sweep's `frames/` directory, beside `scores.csv` and `arms.json`. The directory sits under `{runs}`, keyed by scene, code and arms, so a code change or a new config never reuses stale figures. Figures that already exist are kept; pass `redo=True` after changing `OVERLAY`.

Each figure shows:

- **Top strip:** where the keyframe sits on the timeline (purple line): ADE, heading, start speed, and p(turn).
- **The camera photo** with each view's footprint, the road grid at 10–60 m, and the GT and plans.
- **BEV:** the GT and the plans over 10 s.
- **The model's own road:** lane lines (opacity is their probability), road edges, the lead, and the plan.
- **Speed and heading** per 0.5 s step, and the full **desire_state**.
- **The four model inputs** for each arm (each view at t − 0.2 s and at t), with the expected horizon lines.

In [ ]:
lab.dump(CONFIG, AGAINST, overlay=OVERLAY, workers=WORKERS)

## 6. Frame by frame

Set `FRAME` and re-run the cell. It shows that keyframe's figure (from the part 5 cache, or drawn now if it is missing), then the scores of it and of the two keyframes either side, one column each, its own marked `<<`. `FRAME` takes a keyframe index, a label such as `"dbba76d9/150"`, or a sample id.

With `WIDGETS = True`, `lab.browse()` gives a slider over every keyframe instead.

In [ ]:
FRAME = 150
lab.frame(FRAME, CONFIG, AGAINST, overlay=OVERLAY)

In [ ]:
lab.browse(CONFIG, AGAINST, overlay=OVERLAY)

## 7. What this scene shows (stock, `dbba76d9`)

Read off the cells above on 2026-10-02. Rerunning with another `SCENE`, `START` or `CONFIG` changes the numbers but not this text.

**Measured** (the score table, the scene's poses, and the figures in part 5):

- **The benchmark sees two points of a 12 s story.** It scores 40 (straight: ADE 1.21 m) and 130 (left turn, `not_planned`: ADE 2.32 m). 130 falls in the stretch where the car has slowed for the turn but not yet started it.
- **Where the car turns.** It brakes from 8.2 m/s at keyframe 105 to 3.9 m/s at 132. Its yaw only begins after that: 2.7° at 136, past 5° at 138, past 20° at 145, and about 90° by 180.
    - The 5 s route rule first calls a turn at 110. The GT's 3 s heading passes 10° at 111 and peaks at 62° over 130–134.
    - Keyframes 40–109 are "Continue straight" by the rule, but their 10 s GT already contains the turn. Keyframe 87 is the clearest example.
- **When the model turns.** Its 3 s heading stays under 3° through 134, passes 10° at 138 and 30° at 139, and is `matched` from 140 on (19 keyframes). The 28 keyframes in 110–137 are `not_planned`.
    - Its desire head shows the same timing: p(turn left) is 0.00 until 132, 0.1 at 135, 0.5 at 145, and 0.68 by 159.
    - Its lateral uncertainty at 3 s (`sigma_y_3s`) jumps from under 0.5 m to over 2 m at 135.
- **The straight stretch (40–109) is all longitudinal error.** Mean ADE is 1.92 m, of which lon is 1.91 and lat 0.11.
    - The plan's first step averages 0.87× the pose speed over 40–100.
    - ADE peaks at 4.21 m at 87. There the GT holds 8.2 m/s for 1.5 s, then brakes for the turn about 35 m ahead. The plan starts at 6.3 m/s and brakes from t = 0.
- **The camera height the road head reads** is 1.40 m (median over the 120 keyframes), against the map's 1.93 m. The same low reading appears in the image-tweak notebook.
- **Noise.** Over the 120 keyframes, the largest ADE change any half-pixel shift makes has a median of 0.18 m. It is under 0.3 m on 86% of keyframes and at most 0.76 m (keyframe 107). The straight-stretch hump (up to 4.2 m) and the turn's errors are well above it; single-keyframe wiggles of under 0.5 m are not.

**Argued, not tested:**

- **The model follows this turn; it does not anticipate it.** Its plan and its desire head both move within about 0.3 s of the car's own yaw starting, and neither moves before it. Over 110–134 the intersection is in view, the car is braking hard, and the model's own road edges (part 5's "model's road" panel at 134) already bend left, yet the plan stays straight.
    - That fits a model that reads the turn from frame-to-frame motion in its 4.8 s of history, rather than from the road layout.
    - A `CONFIG` with `desire="turn"` swept over 110–137 is the test. It would show whether being told the turn moves the plan earlier.
- **Most of the error before the turn comes from the start speed, not from the path.** The plan starts slow on every keyframe, and the gap grows where the human holds speed into a late, hard brake (80–95). That matches the earlier finding that ADE is mostly start speed.